In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx

print(jax.__version__)

In [ ]:
# initializing arrays
data = [[1, 2, 3], [4, 5, 6]]
arr1 = jnp.array(data)
print(arr1, arr1.dtype, arr1.shape)

In [ ]:
# from NumPy
np_arr = np.array(data)
arr2 = jnp.array(np_arr)
print(arr2, arr2.dtype, arr2.shape)

In [ ]:
ones = jnp.ones_like(arr2)
print(ones, ones.shape, ones.dtype)
zeros = jnp.zeros_like(arr2)
print(zeros, zeros.shape, zeros.dtype)

In [ ]:
# constants or random values
shape = (3, 3)
ones = jnp.ones(shape)
zeros = jnp.zeros(shape)

seed = 123
key = jax.random.key(seed)
rand = jax.random.uniform(key, shape)
randn = jax.random.normal(key, shape)

print(f"Ones: {ones}")
print(f"Zeros: {zeros}")
print(f"Uniform Random: {rand}")
print(f"Gaussian Random: {randn}")

In [ ]:
randn_2 = jax.random.normal(key, shape)
print((randn == randn_2).all())

In [ ]:
print(key)

In [ ]:
k1, k2 = jax.random.split(key, 2)
print(k1)
print(k2)

In [ ]:
randn_3 = jax.random.normal(k1, shape)
randn_4 = jax.random.normal(k2, shape)
print((randn_3 != randn_4).all())

In [ ]:
# initialize from pytorch
import torch

x_torch = torch.randn(shape)
x_jax = jnp.asarray(x_torch)
print(x_jax, x_jax.shape, x_jax.dtype)

In [ ]:
# without making a copy
x_jax = jax.dlpack.from_dlpack(x_torch, copy=False)
print(x_jax, x_jax.shape, x_jax.dtype)

In [ ]:
print(f"Device: {x_jax.device}")

In [ ]:
try:
    x_jax[0, 0] = 100.0
except TypeError as e:
    print(e)


x_torch = torch.tensor([1, 2, 3, 4])
x_jax = jnp.array([1, 2, 3, 4])
print(f"Default integer dtypes, PyTorch: {x_torch.dtype} and Jax: {x_jax.dtype}")

x_torch = torch.zeros(3, 4)
x_jax = jnp.zeros((3, 4))
print(f"Default float dtypes, PyTorch: {x_torch.dtype} and Jax: {x_jax.dtype}")
print(f"Default devices, PyTorch: {x_torch.device} and Jax: {x_jax.device}")

In [ ]:
# device accelerator stuff
print(f"Available devices {jax.devices('cpu')}")
x_cpu = jax.device_put(x_jax, device=jax.devices("cpu")[0])

In [ ]:
print(x_cpu.device)

In [ ]:
x = jax.random.normal(key, (3, 4))
print(x)

In [ ]:
print(x[:, 0])
print(x[0, :])
print(x[..., 0])

In [ ]:
x = x.at[0, 0].set(0)
print(x)

In [ ]:
print(jnp.arange(10)[-1])

In [ ]:
cat = jnp.concat([x, x], axis=-1)
print(cat)

In [ ]:
y1 = x @ x.T
y2 = jnp.matmul(x, x.T)
print((y1 == y2).all())

In [ ]:
print(y1.shape)

In [ ]:
y1 = x * x
y2 = jnp.multiply(x, x)
print((y1 == y2).all())

In [ ]:
print(y1.shape)

In [ ]:
agg = x.sum()
print(type(agg.item()))

In [ ]:
# learning autodiff

# Input tensor
x = jax.random.normal(k2, (5))
# Target output
y_true = jax.random.normal(k1, (3))

# Initialize random parameters
seed = 123
key = jax.random.key(seed)
key, w_key, b_key = jax.random.split(key, 3)
w = jax.random.normal(w_key, (5, 3))
b = jax.random.normal(b_key, (3,))


# model function
def predict(x, w, b):
    return jnp.matmul(x, w) + b


# Criterion or loss function
def compute_loss(w, b, x, y_true):
    y_pred = predict(x, w, b)
    return jnp.mean((y_true - y_pred) ** 2)


loss = compute_loss(w, b, x, y_true)
print(loss)

In [ ]:
loss, (w_grad, b_grad) = jax.value_and_grad(compute_loss, argnums=(0, 1))(
    w, b, x, y_true
)
print(f"{w_grad=}")
print(f"{b_grad=}")
print(f"{loss=}")

In [ ]:
net_params = {"weights": w, "bias": b}


# Criterion or loss function
def compute_loss_2(param_dict, x, y_true):
    y_pred = predict(x, param_dict["weights"], param_dict["bias"])
    return jnp.mean((y_true - y_pred) ** 2)


loss, grad_dict = jax.value_and_grad(compute_loss_2, argnums=0)(net_params, x, y_true)
print(f"{loss=}")
print(f"{grad_dict=}")

In [ ]:
# To install Flax: `pip install -U flax treescope optax`


class BasicBlock(nnx.Module):
    def __init__(
        self,
        in_planes: int,
        out_planes: int,
        do_downsample: bool = False,
        *,
        rngs: nnx.Rngs,
    ):
        strides = (2, 2) if do_downsample else (1, 1)
        self.conv1_bn1 = nnx.Sequential(
            nnx.Conv(
                in_planes,
                out_planes,
                kernel_size=(3, 3),
                strides=strides,
                padding="SAME",
                use_bias=False,
                rngs=rngs,
            ),
            nnx.BatchNorm(out_planes, momentum=0.9, epsilon=1e-5, rngs=rngs),
        )
        self.conv2_bn2 = nnx.Sequential(
            nnx.Conv(
                out_planes,
                out_planes,
                kernel_size=(3, 3),
                strides=(1, 1),
                padding="SAME",
                use_bias=False,
                rngs=rngs,
            ),
            nnx.BatchNorm(out_planes, momentum=0.9, epsilon=1e-5, rngs=rngs),
        )

        if do_downsample:
            self.conv3_bn3 = nnx.Sequential(
                nnx.Conv(
                    in_planes,
                    out_planes,
                    kernel_size=(1, 1),
                    strides=(2, 2),
                    padding="VALID",
                    use_bias=False,
                    rngs=rngs,
                ),
                nnx.BatchNorm(out_planes, momentum=0.9, epsilon=1e-5, rngs=rngs),
            )
        else:
            self.conv3_bn3 = lambda x: x

    def __call__(self, x: jax.Array):
        out = self.conv1_bn1(x)
        out = nnx.relu(out)

        out = self.conv2_bn2(out)
        out = nnx.relu(out)

        shortcut = self.conv3_bn3(x)
        out += shortcut
        out = nnx.relu(out)
        return out


class ResNet18(nnx.Module):
    def __init__(self, num_classes: int, *, rngs: nnx.Rngs):
        self.num_classes = num_classes
        self.conv1_bn1 = nnx.Sequential(
            nnx.Conv(
                3,
                64,
                kernel_size=(3, 3),
                strides=(1, 1),
                padding="SAME",
                use_bias=False,
                rngs=rngs,
            ),
            nnx.BatchNorm(64, momentum=0.9, epsilon=1e-5, rngs=rngs),
        )
        self.layer1 = nnx.Sequential(
            BasicBlock(64, 64, rngs=rngs),
            BasicBlock(64, 64, rngs=rngs),
        )
        self.layer2 = nnx.Sequential(
            BasicBlock(64, 128, do_downsample=True, rngs=rngs),
            BasicBlock(128, 128, rngs=rngs),
        )
        self.layer3 = nnx.Sequential(
            BasicBlock(128, 256, do_downsample=True, rngs=rngs),
            BasicBlock(256, 256, rngs=rngs),
        )
        self.layer4 = nnx.Sequential(
            BasicBlock(256, 512, do_downsample=True, rngs=rngs),
            BasicBlock(512, 512, rngs=rngs),
        )
        self.fc = nnx.Linear(512, self.num_classes, rngs=rngs)

    def __call__(self, x: jax.Array):
        x = self.conv1_bn1(x)
        x = nnx.relu(x)
        x = nnx.max_pool(x, (2, 2), strides=(2, 2))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = nnx.avg_pool(x, (x.shape[1], x.shape[2]))
        x = x.reshape((x.shape[0], -1))
        x = self.fc(x)
        return x

In [ ]:
model = ResNet18(10, rngs=nnx.Rngs(0))

# Visualize the model architecture
nnx.display(model)

In [ ]:
x = jnp.ones((4, 32, 32, 3))
y_pred = model(x)
y_pred.shape

In [ ]:
# CIFAR10 training/testing datasets setup

from torchvision.datasets import CIFAR10
from torchvision.transforms import v2 as T


def to_np_array(pil_image):
    return np.asarray(pil_image)


def normalize(image):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = image.astype(np.float32) / 255.0
    return (image - mean) / std


train_transforms = T.Compose(
    [
        T.Pad(4),
        T.RandomCrop(32, fill=128),
        T.RandomHorizontalFlip(),
        T.Lambda(to_np_array),
        T.Lambda(normalize),
    ]
)

test_transforms = T.Compose(
    [
        T.Lambda(to_np_array),
        T.Lambda(normalize),
    ]
)

train_dataset = CIFAR10("./data", train=True, download=True, transform=train_transforms)
test_dataset = CIFAR10("./data", train=True, download=False, transform=test_transforms)

In [ ]:
# Data loaders setup
from torch.utils.data import DataLoader

batch_size = 512


def np_arrays_collate_fn(list_of_datapoints):
    list_of_images = [dp[0] for dp in list_of_datapoints]
    list_of_targets = [dp[1] for dp in list_of_datapoints]
    return np.stack(list_of_images, axis=0), np.asarray(list_of_targets)


train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    num_workers=0,
    shuffle=True,
    collate_fn=np_arrays_collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    num_workers=0,
    shuffle=False,
    collate_fn=np_arrays_collate_fn,
)

In [ ]:
# Let us check training dataloader:
trl_iter = iter(train_loader)
batch = next(trl_iter)
print(batch[0].shape, batch[0].dtype, batch[1].shape, batch[1].dtype)

In [ ]:
import optax

learning_rate = 0.005
momentum = 0.9

optimizer = nnx.ModelAndOptimizer(model, optax.adamw(learning_rate, momentum))

In [ ]:
def compute_loss_and_logits(model: nnx.Module, batch):
    logits = model(batch[0])
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=batch[1]
    ).mean()
    return loss, logits


@nnx.jit
def train_step(
    model: nnx.Module, optimizer: nnx.Optimizer, metrics: nnx.MultiMetric, batch
):
    """Train for a single step."""
    # convert numpy arrays to jnp.array on GPU
    x, y_true = jnp.asarray(batch[0]), jnp.asarray(batch[1])

    grad_fn = nnx.value_and_grad(compute_loss_and_logits, has_aux=True)
    (loss, logits), grads = grad_fn(model, (x, y_true))

    metrics.update(loss=loss, logits=logits, labels=y_true)  # In-place updates.

    optimizer.update(grads)  # In-place updates.
    return loss


@nnx.jit
def eval_step(model: nnx.Module, metrics: nnx.MultiMetric, batch):
    # convert numpy arrays to jnp.array on GPU
    x, y_true = jnp.asarray(batch[0]), jnp.asarray(batch[1])

    loss, logits = compute_loss_and_logits(model, (x, y_true))

    metrics.update(loss=loss, logits=logits, labels=y_true)  # In-place updates.

In [ ]:
# Define helper object to compute train/test metrics
metrics = nnx.MultiMetric(
    accuracy=nnx.metrics.Accuracy(),
    loss=nnx.metrics.Average("loss"),
)

metrics_history = {
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": [],
}

In [ ]:
# Start the training

num_epochs = 3

for epoch in range(num_epochs):
    metrics.reset()  # Reset the metrics for the test set.

    model.train()  # Set model to the training mode: e.g. update batch statistics
    for step, batch in enumerate(train_loader):
        loss = train_step(model, optimizer, metrics, batch)

        print(
            f"\r[train] epoch: {epoch + 1}/{num_epochs}, iteration: {step}, batch loss: {loss.item():.4f}",
            end="",
        )
    print("\r", end="")

    for metric, value in metrics.compute().items():  # Compute the metrics.
        metrics_history[f"train_{metric}"].append(value)  # Record the metrics.
    metrics.reset()  # Reset the metrics for the test set.

    # Compute the metrics on the test set after each training epoch.
    model.eval()  # Set model to evaluation model: e.g. use stored batch statistics
    for test_batch in test_loader:
        eval_step(model, metrics, test_batch)

    # Log the test metrics.
    for metric, value in metrics.compute().items():
        metrics_history[f"test_{metric}"].append(value)
    metrics.reset()  # Reset the metrics for the next training epoch.

    print(
        f"[train] epoch: {epoch + 1}/{num_epochs}, "
        f"loss: {metrics_history['train_loss'][-1]:0.4f}, "
        f" accuracy: {metrics_history['train_accuracy'][-1] * 100:0.4f}"
    )
    print(
        f"[test] epoch: {epoch + 1}/{num_epochs}, "
        f"loss: {metrics_history['test_loss'][-1]:0.4f}, "
        f"accuracy: {metrics_history['test_accuracy'][-1] * 100:0.4f}"
        "\n"
    )

In [ ]:
def matmul_relu_add(x, y):
    z = x * y
    return jax.nn.relu(z) + x

In [ ]:
key = jax.random.key(123)
key1, key2 = jax.random.split(key)
x = jax.random.uniform(key1, (2500, 3000))
y = jax.random.uniform(key2, (2500, 3000))

In [ ]:
%%timeit
matmul_relu_add(x, y)

In [ ]:
jit_matmul_relu = jax.jit(matmul_relu_add)
# Warm-up: compile the function
_ = jit_matmul_relu(x, y)

In [ ]:
%%timeit
jit_matmul_relu(x, y)